<a href="https://colab.research.google.com/github/Theresa-Gao-0529/MSSP607/blob/main/week9_image_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Participation Activity: Week 9




In [ ]:
from pathlib import Path
from urllib.parse import quote, urlencode, urlparse
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import csv
import json
import re
import time

SEARCH_TERM = "cats"
MAX_IMAGES = 10
OUTPUT_DIR = Path("downloaded_images")
CSV_PATH = Path("week9_image_metadata.csv")
API_URL = "https://commons.wikimedia.org/w/api.php"
USER_AGENT = "Week9-ImageDownloader/1.0 (educational assignment)"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Search term: {SEARCH_TERM!r}; maximum images: {MAX_IMAGES}")

In [ ]:
def fetch_json(url, params=None, timeout=30):
    """Fetch JSON with a descriptive User-Agent."""
    if params:
        url = f"{url}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def clean_text(text):
    """Turn a Commons file title into readable extracted text."""
    text = re.sub(r"^File:\s*", "", text or "")
    return re.sub(r"[_\s]+", " ", text).strip()


def search_commons(term, limit):
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": term,
        "gsrnamespace": 6,  # namespace 6 = File
        "gsrlimit": min(max(int(limit), 1), 50),
        "prop": "imageinfo",
        "iiprop": "url|size|mime",
        "iiurlwidth": 1600,
        "format": "json",
        "formatversion": 2,
    }
    data = fetch_json(API_URL, params)
    pages = data.get("query", {}).get("pages", [])
    results = []
    for page in pages:
        imageinfo = (page.get("imageinfo") or [{}])[0]
        image_url = imageinfo.get("thumburl") or imageinfo.get("url")
        if not image_url:
            continue
        results.append({
            "title": clean_text(page.get("title", "")),
            "page_url": page.get("canonicalurl") or page.get("fullurl") or f"https://commons.wikimedia.org/wiki/File:{quote(clean_text(page.get('title', '')).replace(' ', '_'))}",
            "image_url": image_url,
            "mime": imageinfo.get("mime", ""),
            "source_width": imageinfo.get("width", ""),
            "source_height": imageinfo.get("height", ""),
        })
    return results[:limit]


def safe_filename(title, image_url, index):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(title).stem).strip("._") or f"image_{index:02d}"
    suffix = Path(urlparse(image_url).path).suffix.lower()
    if suffix not in {".jpg", ".jpeg", ".png", ".gif", ".webp", ".tif", ".tiff"}:
        suffix = ".jpg"
    return f"{index:02d}_{stem[:80]}{suffix}"


def download_image(image_url, destination, timeout=60):
    request = Request(image_url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout) as response:
        data = response.read()
    destination.write_bytes(data)
    return len(data)


In [ ]:
records = []
try:
    matches = search_commons(SEARCH_TERM, MAX_IMAGES)
    print(f"Found {len(matches)} image result(s).")
except (HTTPError, URLError, TimeoutError, ValueError, json.JSONDecodeError) as exc:
    matches = []
    print(f"Search failed: {exc}")

for index, item in enumerate(matches, start=1):
    filename = safe_filename(item["title"], item["image_url"], index)
    destination = OUTPUT_DIR / filename
    row = {
        "search_term": SEARCH_TERM,
        "title": item["title"],
        "page_url": item["page_url"],
        "image_url": item["image_url"],
        "filename": str(destination),
        "mime": item["mime"],
        "source_width": item["source_width"],
        "source_height": item["source_height"],
        "bytes": 0,
        "status": "failed",
        "error": "",
    }
    try:
        row["bytes"] = download_image(item["image_url"], destination)
        row["status"] = "downloaded"
    except (HTTPError, URLError, TimeoutError, OSError) as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
    records.append(row)
    time.sleep(0.2)

fieldnames = ["search_term", "title", "page_url", "image_url", "filename", "mime", "source_width", "source_height", "bytes", "status", "error"]
with CSV_PATH.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

print(f"Wrote {CSV_PATH} with {len(records)} record(s).")
print(f"Downloaded {sum(row['status'] == 'downloaded' for row in records)} image(s) to {OUTPUT_DIR}/")

In [ ]:
with CSV_PATH.open(encoding="utf-8", newline="") as csv_file:
    saved_rows = list(csv.DictReader(csv_file))
print("CSV columns:", fieldnames)
print("CSV rows:", len(saved_rows))
assert CSV_PATH.exists()
assert CSV_PATH.stat().st_size > 0
